Experimenting with generating realistic layouts

In [4]:
import sys
sys.path.append("/Users/janikeuskirchen/Code/cargopal/cargoformer/venv/lib/python3.8/site-packages")

In [8]:
import numpy as np
from squaternion import Quaternion
from typing import List 

In [6]:
NUM_ITEMS = 10
NUM_SIMULATIONS = 1
ORIENTATION_NOISE = 0.01

In [ ]:
def generate_random_item():
    dim = np.random.uniform(0.1, 0.5, size=3).round(4)
    pos = [*np.random.uniform(-3.0, 3.0, size=2).round(4),
           max(np.round(np.random.uniform(0, 3.0), 4), dim[2]/2+0.01)]
    # Making sure that the pos_height is at least the dim_height/2, otherwise the item is stuck in the ground
    # re-order so it's the same ordering as in pybullet; only if I use squaternion above!
    # each item's vector looks like this: [x, y, z, w, d, h, q1, q2, q3, q4, m]
    # [x, y, z] is the position, [w, d, h] is the shape
    # Orientation
    q = np.array(Quaternion.from_euler(0., 0., 0.))
    q = q[[1, 2, 3, 0]]  # re-order so it's the same ordering as in pybullet; only if I use squaternion above!
    # add a little bit of noise to the orientation
    noise = np.random.normal(0, ORIENTATION_NOISE, size=4)
    q = (q + noise).round(4)
    orn = q
    # for now, assume zero orientation (they're all cuboids; I can change orientation by changing its dimensions)
    # for Euler orientation I would use notation [a, b, g] (alpha, beta, gamma)
    # Mass
    # to make things easy, mass is always simply proportional to volume and this proportionality coefficient is
    # randomly sampled to be between, say, 5 and 15, according to a uniform distribution
    m = (np.prod(dim)*np.random.uniform(5, 15)).round(4)
    # Each item's vector looks like this: [x, y, z, w, d, h, q1, q2, q3, q4, m]
    return np.array([*pos, *dim, *orn, m])

In [ ]:
def intersects(item1, item2, margin: float = 0.1):
    # https://gamedev.stackexchange.com/questions/23748/testing-whether-two-cubes-are-touching-in-space
    # True if item1 and item2 intersect
    # Note that w, h, d are half-extents!!
    x1, y1, z1, w1, d1, h1, _, _, _, _, _ = item1
    x2, y2, z2, w2, d2, h2, _, _, _, _, _ = item2
    # Just to make sure that they are also not too close to each other, add a small "margin" to each item
    # only for comparison purposes
    w1, d1, h1 = np.array([w1, d1, h1]) + margin
    w2, d2, h2 = np.array([w2, d2, h2]) + margin
    min_x1, max_x1, min_y1, max_y1, min_z1, max_z1 = x1-w1, x1+w1, y1-d1, y1+d1, z1-h1, z1+h1
    min_x2, max_x2, min_y2, max_y2, min_z2, max_z2 = x2-w2, x2+w2, y2-d2, y2+d2, z2-h2, z2+h2
    return ((min_x1 < min_x2 < max_x1) or (min_x2 < min_x1 < max_x2)) and \
           ((min_y1 < min_y2 < max_y1) or (min_y2 < min_y1 < max_y2)) and \
           ((min_z1 < min_z2 < max_z1) or (min_z2 < min_z1 < max_z2))

In [ ]:
def any_intersects(item1, items, margin: float = 0.1):
    for item in items:
        if intersects(item1, item, margin=margin):
            return True
        return False

In [ ]:
def generate_random_example() -> List:
    items = []
    for _ in range(NUM_ITEMS):
        items.append(generate_random_item())
    return items

In [ ]:
def generate_realistic_example(num_items: int = NUM_ITEMS, margin: float = 0.1) -> List:
    # TODO: Check if it's even possible to have an arrangement with this many items in the expected space
    #  Since the position is sampled from a normal distribution, it might eventually randomly sample a box
    #  that's far enough not to intersect
    items = []
    trying_again_counter = 0
    for _ in range(num_items):
        proposed_item = generate_random_item()
        if i > 0:
            while any_intersects(proposed_item, items, margin=margin):
                print("trying again")
                # trying_again_counter += 1
                proposed_item = generate_random_item()
        items.append(proposed_item)
        # print(proposed_item)
    # return trying_again_counter
    return items

In [216]:
np.array(generate_realistic_example()).shape

(10, 11)

In [212]:
items = generate_realistic_example(30)
for i, item in enumerate(items):
    if i > 0:
        print(any_intersects(item, items[:i]))
        # print(i, list(range(len(items)))[:i])

False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
